In [1]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Model 
from tensorflow.keras.models import Sequential
from tensorflow import math
import numpy

# Setting random seeds
numpy.random.seed(10)

In [6]:
vocab_size = 500
model_dimension = 128

# Define the LSTM model
LSTM = Sequential()
LSTM.add(layers.Embedding(input_dim=vocab_size, output_dim=model_dimension))
LSTM.add(layers.LSTM(units=model_dimension, return_sequences=True))
LSTM.add(layers.GlobalAveragePooling1D())
LSTM.add(layers.Lambda(lambda x: tf.math.l2_normalize(x)))

input1 = layers.Input((None,))
input2 = layers.Input((None,))

# Concatenate two LSTMs together
conc = layers.Concatenate(axis=1)((LSTM(input1), LSTM(input2)))

# Use the parallel combinator to create a Siamese model out of the LSTM
Siamese = Model(inputs=(input1, input2), outputs=conc)

# Print the summary of the model
Siamese.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_4       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_4        │ (None, 128)       │    195,584 │ input_layer_3[0]… │
│ (Sequential)        │                   │            │ input_layer_4[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 256)       │          0 │ sequential_4[0][… │
│ (Concatenate)       │                   │            │ sequential_4[1][… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 195,584 (764.00 KB)

 Trainable params: 195,584 (764.00 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
def show_layers(model, layer_prefix):
    print (f"Total layers: {len(model.layers)}\n")
    for i in range (len(model.layers)):
        print ('====================')
        print(f'{layer_prefix}_{i}: {model.layers[i]}\n')

print('Siamese model: \n')
show_layers (Siamese, 'parallel.sub-layers')

print('Detail of LSTM models:\n')
show_layers(LSTM,'Serial.sub-layers' ) 

Siamese model: 

Total layers: 4

parallel.sub-layers_0: <InputLayer name=input_layer_3, built=True>

parallel.sub-layers_1: <InputLayer name=input_layer_4, built=True>

parallel.sub-layers_2: <Sequential name=sequential_4, built=True>

parallel.sub-layers_3: <Concatenate name=concatenate_1, built=True>

Detail of LSTM models:

Total layers: 4

Serial.sub-layers_0: <Embedding name=embedding_4, built=True>

Serial.sub-layers_1: <LSTM name=lstm_4, built=True>

Serial.sub-layers_2: <GlobalAveragePooling1D name=global_average_pooling1d_1, built=True>

Serial.sub-layers_3: <Lambda name=lambda_1, built=True>

